In [1]:
import MDAnalysis as mda
import pandas as pd
from biopandas.pdb import PandasPdb
import os
import glob
import re
import math
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw

def pdb_to_dataframe(pdb_file):
    """
    Load a PDB file using MDAnalysis and convert key atom information to a pandas DataFrame.
    """
    u = mda.Universe(pdb_file)
    
    # Extract atom-related data: atom name, residue name, residue ID, and chain ID
    atom_data = {
        'Atom Name': u.atoms.names,
        'Residue Name': u.atoms.resnames,
        'Residue ID': u.atoms.resids,
        'Chain ID': u.atoms.segids,
        'X': u.atoms.positions[:, 0],
        'Y': u.atoms.positions[:, 1],
        'Z': u.atoms.positions[:, 2],
    }
    
    # Create a pandas DataFrame from the atom data
    df = pd.DataFrame(atom_data)
    
    return df

def grid_list(atom_df):
    return list(zip(atom_df['x_coord'], atom_df['y_coord'], atom_df['z_coord']))

def filtering_proteins(atom_df, grid_list, radius=5.0):
    import numpy as np

    # Adjust these column names if needed for your dataframe
    residue_cols = ['chain_id', 'residue_name', 'residue_number', 'insertion']
    residue_cols = [col for col in residue_cols if col in atom_df.columns]

    if not residue_cols:
        raise ValueError("Could not identify residue columns in atom_df.")

    atom_coords = atom_df[['x_coord', 'y_coord', 'z_coord']].values
    initially_filtered_atoms = set()

    # Step 1: find atoms within radius of any grid point
    for x, y, z in grid_list:
        distances_sq = (
            (atom_coords[:, 0] - x) ** 2 +
            (atom_coords[:, 1] - y) ** 2 +
            (atom_coords[:, 2] - z) ** 2
        )
        mask = distances_sq <= radius ** 2
        initially_filtered_atoms.update(atom_df.index[mask])

    print(f"Total atoms within {radius} Å cutoff: {len(initially_filtered_atoms)}")

    if not initially_filtered_atoms:
        return atom_df.loc[list(initially_filtered_atoms)]

    grouped = atom_df.groupby(residue_cols)

    # Step 2: residues passing the 50% rule
    residue_keep_set = set()

    for residue_key, residue_df in grouped:
        residue_atom_indices = set(residue_df.index)
        n_total = len(residue_atom_indices)
        n_filtered = len(residue_atom_indices & initially_filtered_atoms)

        if n_total == 0:
            continue

        fraction_present = n_filtered / n_total

        if fraction_present >= 0.5:
            residue_keep_set.add(residue_key)

    # Step 3: expand to all atoms in kept residues
    filtered_atoms = set()
    for residue_key, residue_df in grouped:
        if residue_key in residue_keep_set:
            filtered_atoms.update(residue_df.index)

    # Fallback: if nothing survives 50% rule, keep residues that had any atom in cutoff
    if len(filtered_atoms) == 0:
        print("No residues passed the 50% occupancy filter. Falling back to residues with at least one atom within cutoff.")

        fallback_residue_keep_set = set()

        for residue_key, residue_df in grouped:
            residue_atom_indices = set(residue_df.index)
            if len(residue_atom_indices & initially_filtered_atoms) > 0:
                fallback_residue_keep_set.add(residue_key)

        for residue_key, residue_df in grouped:
            if residue_key in fallback_residue_keep_set:
                filtered_atoms.update(residue_df.index)

    print(f"Total atoms after residue expansion + filtering: {len(filtered_atoms)}")
    return atom_df.loc[list(filtered_atoms)]


/home/alexhernandez/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_fpocket_protein_name(filename):
    basename = os.path.basename(filename)
    match = re.match(r'([a-zA-Z0-9]{4})', basename)
    if match:
        return match.group(1).upper()
    return None

def get_fpocket_pockets(fpocket_file):
    """
    Read an fpocket output PDB and return:
        pocket_id -> DataFrame of fpocket HETATM pseudo-atoms for that pocket
    """
    ppdb = PandasPdb().read_pdb(fpocket_file)
    hetatm = ppdb.df['HETATM'].copy()

    if hetatm.empty:
        return {}

    hetatm = hetatm[~hetatm['atom_name'].astype(str).str.startswith('H')].copy()

    pockets = {}
    for pocket_id in sorted(hetatm['residue_number'].dropna().unique()):
        pocket_df = hetatm[hetatm['residue_number'] == pocket_id].copy()
        if not pocket_df.empty:
            pockets[int(pocket_id)] = pocket_df

    return pockets

def get_protein_name(filename):
    basename = os.path.basename(filename)
    match = re.match(r'([a-zA-Z0-9]{4})', basename)
    if match:
        return match.group(1).upper()
    return None

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

In [3]:
def get_positive_ligand_atoms(positive_file, protein_name):
    protein_pdb_df = PandasPdb().read_pdb(positive_file)
    protein_pdb_df.df.keys()
    protein = protein_pdb_df.df['ATOM']
    protein = protein[~protein['atom_name'].str.startswith('H')] # don't use hydrogen
    protein_coords = protein[['x_coord', 'y_coord', 'z_coord']].values
    protein_centroid = protein_coords.mean(axis=0)
    print(set(protein['chain_id']))
    print(positive_file)

    ligand_df = PandasPdb().read_pdb(positive_file)
    ligand_df.df.keys()
    ligand = ligand_df.df['HETATM']
    ligand = ligand[ligand['residue_name']=="CLR"]
    x = list(set(zip(ligand['residue_number'], ligand['chain_id'])))

    #get the most inward residue
    min_distance = float('inf')
    closest_clr = None

    all_ligands = []

    for residue_number, chain_id in x:
        clr_atoms = ligand[(ligand['residue_number'] == residue_number) & (ligand['chain_id'] == chain_id)]
        if clr_atoms.empty:
            continue

        clr_coords = clr_atoms[['x_coord', 'y_coord', 'z_coord']].values
        clr_centroid = clr_coords.mean(axis=0)
        
        distance = np.linalg.norm(protein_centroid - clr_centroid)
        
        if distance < min_distance:
            min_distance = distance
            closest_clr = (residue_number, chain_id)

        grid_list_ = grid_list(clr_atoms)

        all_ligands.append(grid_list_)

    ligand_ = ligand[(ligand['residue_number'] == closest_clr[0]) & (ligand['chain_id'] == closest_clr[1])]
    grid_list_ = grid_list(ligand_)

    filtered_atoms = filtering_proteins(protein, grid_list_)

    # Save to pdb
    filtered_pdb = PandasPdb()
    filtered_pdb.df['ATOM'] = filtered_atoms
    filtered_pdb_path = f"filtered-rdkit-fpocket-5A/positive/{protein_name}-filtered.pdb"
    os.makedirs(os.path.dirname(filtered_pdb_path), exist_ok=True)
    filtered_pdb.to_pdb(path=filtered_pdb_path, records=None, gz=False, append_newline=True)
    print(f"Saved: {filtered_pdb_path}")

    return protein, all_ligands


In [4]:
def check_if_unlabeled_is_positive(positive_grid_list, unlabeled_grid_list, cutoff=5.0):
    positive_grid_list = np.asarray(positive_grid_list, dtype=float)
    unlabeled_grid_list = np.asarray(unlabeled_grid_list, dtype=float)

    if positive_grid_list.size == 0 or unlabeled_grid_list.size == 0:
        print("Empty positive_grid_list or unlabeled_grid_list")
        return False

    chol_centroid = positive_grid_list.mean(axis=0)
    fpocket_centroid = unlabeled_grid_list.mean(axis=0)

    distance = np.linalg.norm(chol_centroid - fpocket_centroid)
    print(f"Centroid distance: {distance:.3f} Å")
    return distance <= cutoff

In [5]:
def get_protein_name(filename):
    basename = os.path.basename(filename)  # Get file name without path
    match = re.match(r'([a-zA-Z0-9]{4})', basename)  # Match the first 4-character PDB ID
    if match:
        return match.group(1).upper()
    else:
        return None
def get_mode_index(filename):
    basename = os.path.basename(filename)
    match = re.search(r'mode_(\d+)', basename)
    if match:
        return int(match.group(1))
    else:
        return None  # or raise ValueError("No mode index found.")

def natural_sort_key(s):
    """Function to sort strings in a natural alphanumeric order."""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]


In [6]:
# positive_files = glob.glob("../GNN/CLR-PDB/*.pdb")
# positive_files = sorted(positive_files, key=natural_sort_key)

# fpocket_files = glob.glob("../FPocketUnlabeledUnfiltered/*/*_protein_out.pdb")
# fpocket_files = sorted(fpocket_files, key=natural_sort_key)

# positive_index = 0
# protein, all_lig_gridlist = get_positive_ligand_atoms(
#     positive_files[positive_index],
#     get_protein_name(positive_files[positive_index])
# )

# for fpocket_file in fpocket_files:
#     positive_name = get_protein_name(positive_files[positive_index])
#     unlabeled_name = get_fpocket_protein_name(fpocket_file)

#     if unlabeled_name is None:
#         print(f"Skipping file with no PDB id match: {fpocket_file}")
#         continue

#     if positive_name != unlabeled_name:
#         positive_index += 1

#         if positive_index >= len(positive_files):
#             raise Exception(f"Ran out of positive files while matching {fpocket_file}")

#         positive_name = get_protein_name(positive_files[positive_index])

#         if positive_name != unlabeled_name:
#             raise Exception(
#                 f"Proteins Not Matching Up!!! positive={positive_name}, fpocket={unlabeled_name}"
#             )

#         protein, all_lig_gridlist = get_positive_ligand_atoms(
#             positive_files[positive_index],
#             positive_name
#         )

#     pockets = get_fpocket_pockets(fpocket_file)

#     if not pockets:
#         print(f"No pockets found in {fpocket_file}")
#         continue

#     for pocket_id, pocket_df in pockets.items():
#         grid_list_ = grid_list(pocket_df)

#         # this is the NEW filtering logic being applied to RAW fpocket pockets
#         filtered_atoms = filtering_proteins(protein, grid_list_)

#         if filtered_atoms.empty:
#             continue

#         is_positive = False
#         for lig in all_lig_gridlist:
#             if check_if_unlabeled_is_positive(lig.copy(), grid_list_.copy()):
#                 is_positive = True
#                 break

#         filtered_pdb = PandasPdb()
#         filtered_pdb.df['ATOM'] = filtered_atoms

#         if is_positive:
#             filtered_pdb_path = f"filtered-rdkit-fpocket-5A/unlabeled/{unlabeled_name}-p{pocket_id}-positive.pdb"
#         else:
#             filtered_pdb_path = f"filtered-rdkit-fpocket-5A/unlabeled/{unlabeled_name}-p{pocket_id}.pdb"

#         os.makedirs(os.path.dirname(filtered_pdb_path), exist_ok=True)
#         filtered_pdb.to_pdb(path=filtered_pdb_path, records=None, gz=False, append_newline=True)

#         print(f"Saved: {filtered_pdb_path}")

In [7]:
# %%
import os
import glob
import re
from io import StringIO

import py3Dmol
import pandas as pd
from biopandas.pdb import PandasPdb


def _df_to_pdb_block(df, record_name="ATOM"):
    """
    Convert a Biopandas-style ATOM/HETATM dataframe to a PDB block string.
    """
    if df is None or df.empty:
        return ""

    lines = []
    for _, row in df.iterrows():
        atom_number = int(row.get("atom_number", 1))
        atom_name = str(row.get("atom_name", "X"))[:4]
        alt_loc = str(row.get("alt_loc", ""))[:1]
        residue_name = str(row.get("residue_name", "UNK"))[:3]
        chain_id = str(row.get("chain_id", "A"))[:1]
        residue_number = int(row.get("residue_number", 1))
        insertion = str(row.get("insertion", ""))[:1]
        x = float(row.get("x_coord", 0.0))
        y = float(row.get("y_coord", 0.0))
        z = float(row.get("z_coord", 0.0))
        occupancy = float(row.get("occupancy", 1.00))
        b_factor = float(row.get("b_factor", 0.00))
        element_symbol = str(row.get("element_symbol", "")).strip()

        if not element_symbol:
            # simple fallback from atom name
            cleaned = re.sub(r'[^A-Za-z]', '', atom_name).strip()
            element_symbol = cleaned[:2].capitalize() if cleaned else "C"

        charge = str(row.get("charge", "")).strip()

        line = (
            f"{record_name:<6}"
            f"{atom_number:>5} "
            f"{atom_name:<4}"
            f"{alt_loc:1}"
            f"{residue_name:>3} "
            f"{chain_id:1}"
            f"{residue_number:>4}"
            f"{insertion:1}   "
            f"{x:>8.3f}"
            f"{y:>8.3f}"
            f"{z:>8.3f}"
            f"{occupancy:>6.2f}"
            f"{b_factor:>6.2f}          "
            f"{element_symbol:>2}"
            f"{charge:>2}"
        )
        lines.append(line)

    lines.append("END")
    return "\n".join(lines)


def _find_positive_file(pdb_id, positive_glob="../GNN/CLR-PDB/*.pdb"):
    pdb_id = pdb_id.upper()
    matches = sorted(glob.glob(positive_glob))
    for f in matches:
        base = os.path.basename(f)
        if re.match(rf"{pdb_id}", base, re.IGNORECASE):
            return f
    return None


def _find_fpocket_file(pdb_id, fpocket_glob="../FPocketUnlabeledUnfiltered/*/*_protein_out.pdb"):
    pdb_id = pdb_id.upper()
    matches = sorted(glob.glob(fpocket_glob))
    for f in matches:
        base = os.path.basename(f)
        if re.match(rf"{pdb_id}", base, re.IGNORECASE):
            return f
    return None


def get_all_cholesterol_atoms(positive_file):
    """
    Return:
      protein_df: ATOM dataframe without hydrogens
      chol_groups: list of HETATM dataframes, one per CLR molecule
    """
    ppdb = PandasPdb().read_pdb(positive_file)

    protein_df = ppdb.df["ATOM"].copy()
    protein_df = protein_df[~protein_df["atom_name"].astype(str).str.startswith("H")].copy()

    hetatm = ppdb.df["HETATM"].copy()
    chol_df = hetatm[hetatm["residue_name"] == "CLR"].copy()
    chol_df = chol_df[~chol_df["atom_name"].astype(str).str.startswith("H")].copy()

    chol_groups = []
    for (resnum, chain_id), group in chol_df.groupby(["residue_number", "chain_id"], dropna=False):
        if not group.empty:
            chol_groups.append(group.copy())

    return protein_df, chol_groups


def build_filtered_region_from_grid(protein_df, grid_points, radius=5.0):
    """
    Reuse your existing filtering_proteins() function.
    grid_points should be [(x,y,z), ...]
    """
    return filtering_proteins(protein_df.copy(), grid_points, radius=radius)


def show_cholesterol_fpocket_view(
    pdb_id,
    pocket_ids=None,
    show_full_protein=True,
    show_cholesterol=True,
    show_fpocket=True,
    show_filtered_chol_regions=True,
    show_filtered_fpocket_regions=True,
    filter_radius=5.0,
    width=1100,
    height=800,
    background="white",
):
    """
    Visualize:
      - full protein
      - all experimental cholesterol molecules
      - all fpocket alpha spheres (or selected pockets)
      - filtered protein regions around cholesterol
      - filtered protein regions around fpocket pockets

    pocket_ids:
      None -> all pockets
      int  -> one pocket
      list -> selected pockets
    """
    pdb_id = pdb_id.upper()

    positive_file = _find_positive_file(pdb_id)
    fpocket_file = _find_fpocket_file(pdb_id)

    if positive_file is None:
        raise FileNotFoundError(f"Could not find experimental structure for {pdb_id}")
    if fpocket_file is None:
        raise FileNotFoundError(f"Could not find fpocket output for {pdb_id}")

    protein_df, chol_groups = get_all_cholesterol_atoms(positive_file)
    fpocket_pockets = get_fpocket_pockets(fpocket_file)

    if isinstance(pocket_ids, int):
        pocket_ids = [pocket_ids]

    if pocket_ids is not None:
        fpocket_pockets = {pid: df for pid, df in fpocket_pockets.items() if pid in pocket_ids}

    view = py3Dmol.view(width=width, height=height)
    view.setBackgroundColor(background)

    # Model 0: full protein
    if show_full_protein:
        protein_block = _df_to_pdb_block(protein_df, record_name="ATOM")
        view.addModel(protein_block, "pdb")
        view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray"}})

    model_index = 1

    # Cholesterol molecules
    if show_cholesterol:
        for chol_df in chol_groups:
            chol_block = _df_to_pdb_block(chol_df, record_name="HETATM")
            view.addModel(chol_block, "pdb")
            view.setStyle({"model": model_index}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.18}})
            model_index += 1

    # fpocket alpha spheres
    if show_fpocket:
        for pocket_id, pocket_df in fpocket_pockets.items():
            pocket_block = _df_to_pdb_block(pocket_df, record_name="HETATM")
            view.addModel(pocket_block, "pdb")
            # sphere is the closest py3Dmol style for a VDW-like alpha-sphere display
            view.setStyle({"model": model_index}, {"sphere": {"color": "red", "opacity": 0.9, "scale": 0.9}})
            model_index += 1

    # Filtered regions around cholesterol
    if show_filtered_chol_regions:
        for chol_df in chol_groups:
            chol_grid = grid_list(chol_df)
            filtered_df = build_filtered_region_from_grid(protein_df, chol_grid, radius=filter_radius)
            if filtered_df is not None and not filtered_df.empty:
                filtered_block = _df_to_pdb_block(filtered_df, record_name="ATOM")
                view.addModel(filtered_block, "pdb")
                view.setStyle(
                    {"model": model_index},
                    {"sphere": {"color": "cyan", "opacity": 0.75, "scale": 0.55}}
                )
                model_index += 1

    # Filtered regions around fpocket pockets
    if show_filtered_fpocket_regions:
        for pocket_id, pocket_df in fpocket_pockets.items():
            pocket_grid = grid_list(pocket_df)
            filtered_df = build_filtered_region_from_grid(protein_df, pocket_grid, radius=filter_radius)
            if filtered_df is not None and not filtered_df.empty:
                filtered_block = _df_to_pdb_block(filtered_df, record_name="ATOM")
                view.addModel(filtered_block, "pdb")
                view.setStyle(
                    {"model": model_index},
                    {"sphere": {"color": "magenta", "opacity": 0.75, "scale": 0.55}}
                )
                model_index += 1

    view.zoomTo()
    return view

In [8]:
show_cholesterol_fpocket_view("1ZHY", pocket_ids=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

Total atoms within 5.0 Å cutoff: 77
Total atoms after residue expansion + filtering: 49
Total atoms within 5.0 Å cutoff: 207
Total atoms after residue expansion + filtering: 196
Total atoms within 5.0 Å cutoff: 60
Total atoms after residue expansion + filtering: 59
Total atoms within 5.0 Å cutoff: 115
Total atoms after residue expansion + filtering: 83
Total atoms within 5.0 Å cutoff: 52
Total atoms after residue expansion + filtering: 55
Total atoms within 5.0 Å cutoff: 36
Total atoms after residue expansion + filtering: 29
Total atoms within 5.0 Å cutoff: 58
Total atoms after residue expansion + filtering: 62
Total atoms within 5.0 Å cutoff: 80
Total atoms after residue expansion + filtering: 86
Total atoms within 5.0 Å cutoff: 34
Total atoms after residue expansion + filtering: 39
Total atoms within 5.0 Å cutoff: 43
Total atoms after residue expansion + filtering: 51
Total atoms within 5.0 Å cutoff: 48
Total atoms after residue expansion + filtering: 56


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
import os
import re
import glob
import tempfile
import subprocess
import numpy as np
import pandas as pd
import MDAnalysis as mda
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import rdchem

# -------------------------
# Atom subtype setup
# -------------------------
ATOM_SUBTYPES = [
    # Carbon (17)
    'C', 'CA', 'CB', 'CD', 'CD1', 'CD2', 'CE', 'CE1', 'CE2', 'CE3',
    'CG', 'CG1', 'CG2', 'CH2', 'CZ', 'CZ2', 'CZ3',

    # Oxygen (8)
    'O', 'OH', 'OD1', 'OD2', 'OE1', 'OE2', 'OG', 'OG1',

    # Nitrogen (9)
    'N', 'NE', 'NE1', 'NE2', 'ND1', 'ND2', 'NZ', 'NH1', 'NH2',

    # Sulfur (2)
    'SD', 'SG',

    # Unknown (1)
    'UNKNOWN'
]

ATOM_TO_INDEX = {atom: idx for idx, atom in enumerate(ATOM_SUBTYPES)}

# residue groups
ACIDIC = {'ASP', 'GLU'}
BASIC = {'LYS', 'ARG'}
HIS = {'HIS'}
CYS = {'CYS'}
POLAR_UNCHARGED = {'ASN', 'GLN', 'SER', 'THR'}
GLY = {'GLY'}
PRO = {'PRO'}
AROMATIC = {'PHE', 'TYR', 'TRP'}
HYDROPHOBIC = {'ALA', 'ILE', 'LEU', 'MET', 'VAL'}

# 37 subtype + 3 hybrid + 3 misc(partial_charge, ring, aromatic) + 9 residue = 52
N_ATOM_SUBTYPE = len(ATOM_SUBTYPES)
N_HYBRID = 3
N_MISC = 3
N_RESIDUE_GROUPS = 9
ENCODING_SIZE = N_ATOM_SUBTYPE + N_HYBRID + N_MISC + N_RESIDUE_GROUPS

PDB2PQR_PH = 7.4


def one_hot_encoding(pdb_df, dtype=np.int8):
    num_rows = len(pdb_df)
    one_hot_matrix = np.zeros((num_rows, N_ATOM_SUBTYPE), dtype=dtype)

    for i, atom_name in enumerate(pdb_df['Atom Name']):
        atom_name = str(atom_name).strip()
        idx = ATOM_TO_INDEX.get(atom_name, ATOM_TO_INDEX['UNKNOWN'])
        one_hot_matrix[i, idx] = 1

    return one_hot_matrix


def pdb_to_dataframe(pdb_file):
    u = mda.Universe(pdb_file)
    atom_data = {
        'Atom Name': u.atoms.names,
        'Residue Name': u.atoms.resnames,
        'Residue ID': u.atoms.resids,
        'Chain ID': u.atoms.segids,
        'X': u.atoms.positions[:, 0],
        'Y': u.atoms.positions[:, 1],
        'Z': u.atoms.positions[:, 2],
    }
    return pd.DataFrame(atom_data)

# -------------------------
# pdb2pqr charge helpers
# -------------------------

def strip_dum_atoms(pdb_path, out_path):
    with open(pdb_path) as f_in, open(out_path, "w") as f_out:
        for line in f_in:
            if "DUM" not in line:
                f_out.write(line)


def run_pdb2pqr(clean_pdb, pqr_out, ph=7.4):
    cmd = [
        "pdb2pqr",
        "--ff=AMBER",
        f"--with-ph={ph}",
        "--quiet",
        str(clean_pdb),
        str(pqr_out),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0


def parse_pqr_charges(pqr_path):
    """
    Returns:
        charges_full[(chain, resseq, atomname)] = charge
        charges_nochain[(resseq, atomname)] = charge
        has_chain = bool
    """
    charges_full = {}
    charges_nochain = {}
    chain_values = set()

    with open(pqr_path) as f:
        for line in f:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            try:
                atomname = line[12:16].strip()
                chain = line[21].strip()
                resseq = int(line[22:26].strip())
                charge = float(line[54:62].strip())
            except (ValueError, IndexError):
                continue

            chain_values.add(chain)
            charges_full[(chain, resseq, atomname)] = charge
            charges_nochain[(resseq, atomname)] = charge

    has_chain = any(c != "" for c in chain_values)
    return charges_full, charges_nochain, has_chain


def lookup_charge(chain, resseq, atomname, charges_full, charges_nochain, has_chain):
    # try chain-aware first
    if has_chain:
        val = charges_full.get((chain, resseq, atomname), None)
        if val is not None:
            return val

    # always try no-chain fallback too
    val = charges_nochain.get((resseq, atomname), None)
    if val is not None:
        return val

    return 0.0


def build_pqr_cache(full_pdb_dir, ph=7.4):
    full_pdb_dir = Path(full_pdb_dir)
    pdb_files = sorted(full_pdb_dir.glob("*.pdb"))

    pqr_cache = {}
    pqr_failures = []

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)

        for pdb_path in pdb_files:
            stem = pdb_path.stem
            match = re.match(r'([A-Za-z0-9]{4})', stem)
            if not match:
                pqr_failures.append(stem.upper())
                continue

            pdbid = match.group(1).upper()

            clean_pdb = tmpdir / f"{pdbid}_clean.pdb"
            pqr_out = tmpdir / f"{pdbid}.pqr"

            strip_dum_atoms(pdb_path, clean_pdb)

            ok = run_pdb2pqr(clean_pdb, pqr_out, ph=ph)
            if not ok or not pqr_out.exists():
                pqr_failures.append(pdbid)
                continue

            charges_full, charges_nochain, has_chain = parse_pqr_charges(pqr_out)
            pqr_cache[pdbid] = (charges_full, charges_nochain, has_chain)

    print(f"pdb2pqr complete: {len(pqr_cache)} ok, {len(pqr_failures)} failed")
    if pqr_failures:
        print("Failed PDB IDs:", pqr_failures[:20])

    return pqr_cache, set(pqr_failures)
def rdkit_coords_and_encoding(pdb_path, pdb_df, pqr_cache=None, output_dtype=np.float32):
    """
    Returns:
        coords: (N, 3) float32
        encoded_atoms: (N, 52) float32

    First 37 columns = atom subtype one-hot
    Remaining columns:
        37-39 = hybridization SP/SP2/SP3
        40    = partial_charge (from pdb2pqr)
        41    = is_in_ring
        42    = is_aromatic
        43-51 = residue groups
    """
    mol = Chem.MolFromPDBFile(pdb_path, removeHs=False, sanitize=False)
    if mol is None:
        print(f"[SKIP] RDKit could not parse: {pdb_path}")
        return None, None

    try:
        Chem.SanitizeMol(mol)
    except Exception:
        print(f"[WARN] RDKit sanitize warning for: {pdb_path}")

    conf = mol.GetConformer()

    # exclude hydrogens to match your filtered PDB usage
    heavy_atom_indices = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetAtomicNum() != 1]
    num_atoms = len(heavy_atom_indices)

    if len(pdb_df) != num_atoms:
        print(
            f"[WARNING] Atom count mismatch for {pdb_path}: "
            f"pdb_df has {len(pdb_df)} rows, RDKit non-H atoms has {num_atoms}"
        )
        return None, None

    subtype_one_hot = one_hot_encoding(pdb_df, dtype=np.int8)

    coords = np.zeros((num_atoms, 3), dtype=np.float32)
    encoded_atoms = np.zeros((num_atoms, ENCODING_SIZE), dtype=np.float32)
    encoded_atoms[:, :N_ATOM_SUBTYPE] = subtype_one_hot

    # guess pdb id from filename start, like 1ZHY-... or 1ZHY__...
    base_name = os.path.basename(pdb_path)
    match = re.match(r'([A-Za-z0-9]{4})', base_name)
    pdb_id = match.group(1).upper() if match else None

    charges_full, charges_nochain, has_chain = ({}, {}, False)
    if pqr_cache is not None and pdb_id in pqr_cache:
        charges_full, charges_nochain, has_chain = pqr_cache[pdb_id]

    write_i = 0
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 1:
            continue

        pos = conf.GetAtomPosition(atom.GetIdx())
        coords[write_i] = [pos.x, pos.y, pos.z]

        base = N_ATOM_SUBTYPE

        # hybridization
        hybridization = atom.GetHybridization()
        if hybridization == Chem.HybridizationType.SP:
            encoded_atoms[write_i, base + 0] = 1.0
        elif hybridization == Chem.HybridizationType.SP2:
            encoded_atoms[write_i, base + 1] = 1.0
        elif hybridization == Chem.HybridizationType.SP3:
            encoded_atoms[write_i, base + 2] = 1.0

        # residue / atom info for charge lookup
        pdb_info = atom.GetPDBResidueInfo()
        if pdb_info is not None:
            residue = pdb_info.GetResidueName().strip()
            atomname = pdb_info.GetName().strip()
            chain = (pdb_info.GetChainId() or "").strip()
            resseq = int(pdb_info.GetResidueNumber())
        else:
            residue = ""
            atomname = atom.GetSymbol()
            chain = ""
            resseq = -9999

        partial_charge = lookup_charge(
            chain, resseq, atomname,
            charges_full, charges_nochain, has_chain
        )

        # misc features
        encoded_atoms[write_i, base + 3] = float(partial_charge)
        encoded_atoms[write_i, base + 4] = 1.0 if atom.IsInRing() else 0.0
        encoded_atoms[write_i, base + 5] = 1.0 if atom.GetIsAromatic() else 0.0

        # residue grouping
        if residue in ['ASP', 'GLU']:
            encoded_atoms[write_i, base + 6] = 1.0
        elif residue in ['LYS', 'ARG']:
            encoded_atoms[write_i, base + 7] = 1.0
        elif residue == 'HIS':
            encoded_atoms[write_i, base + 8] = 1.0
        elif residue == 'CYS':
            encoded_atoms[write_i, base + 9] = 1.0
        elif residue in ['ASN', 'GLN', 'SER', 'THR']:
            encoded_atoms[write_i, base + 10] = 1.0
        elif residue == 'GLY':
            encoded_atoms[write_i, base + 11] = 1.0
        elif residue == 'PRO':
            encoded_atoms[write_i, base + 12] = 1.0
        elif residue in ['PHE', 'TYR', 'TRP']:
            encoded_atoms[write_i, base + 13] = 1.0
        elif residue in ['ALA', 'ILE', 'LEU', 'MET', 'VAL']:
            encoded_atoms[write_i, base + 14] = 1.0

        write_i += 1

    return coords.astype(np.float32), encoded_atoms.astype(output_dtype)

def compute_inverse_pairwise_distances_from_coords(coords):
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    distances = np.sqrt(np.sum(diff ** 2, axis=-1))

    with np.errstate(divide='ignore'):
        inverse_distances = 1.0 / distances

    np.fill_diagonal(inverse_distances, 1.0)
    inverse_distances = np.minimum(inverse_distances, 1.0)

    return inverse_distances

In [ ]:
import pickle
import os

cache_path = "pqr_cache.pkl"
FULL_PDB_DIR = "/home/alexhernandez/CholBindNet/GNN/CLR-PDB"   # change this

if os.path.exists(cache_path):
    with open(cache_path, "rb") as f:
        data = pickle.load(f)
        pqr_cache = data["pqr_cache"]
        pqr_failures = data["pqr_failures"]

    print(f"Loaded PQR cache from {cache_path}")
    print(f"Cached PDBs: {len(pqr_cache)}")
else:
    print("Cache not found, rebuilding...")
    pqr_cache, pqr_failures = build_pqr_cache(FULL_PDB_DIR, ph=PDB2PQR_PH)

pdb2pqr complete: 763 ok, 7 failed
Failed PDB IDs: ['6IIV', '6WGT', '6WH4', '8JCW', '8JCY', '8SR9', '8SRA']


In [ ]:
# import pickle

# cache_path = "pqr_cache.pkl"

# with open(cache_path, "wb") as f:
#     pickle.dump({
#         "pqr_cache": pqr_cache,
#         "pqr_failures": pqr_failures
#     }, f)

# print(f"Saved PQR cache to {cache_path}")

Saved PQR cache to pqr_cache.pkl


In [11]:
print("num cached pdbs:", len(pqr_cache))
print("num failures:", len(pqr_failures))

example_key = next(iter(pqr_cache))
print("example pdb id:", example_key)

charges_full, charges_nochain, has_chain = pqr_cache[example_key]
print("has_chain:", has_chain)
print("num full charge entries:", len(charges_full))
print("num no-chain charge entries:", len(charges_nochain))

print("sample full keys:", list(charges_full.items())[:10])
print("sample no-chain keys:", list(charges_nochain.items())[:10])

num cached pdbs: 763
num failures: 7
example pdb id: 1LRI
has_chain: False
num full charge entries: 1432
num no-chain charge entries: 1432
sample full keys: [(('', 1, 'N'), 0.1812), (('', 1, 'CA'), 0.0034), (('', 1, 'C'), 0.6163), (('', 1, 'O'), -0.5722), (('', 1, 'CB'), 0.4514), (('', 1, 'OG1'), -0.6764), (('', 1, 'CG2'), -0.2554), (('', 1, 'H'), 0.1934), (('', 1, 'HA'), 0.1087), (('', 1, 'HB'), -0.0323)]
sample no-chain keys: [((1, 'N'), 0.1812), ((1, 'CA'), 0.0034), ((1, 'C'), 0.6163), ((1, 'O'), -0.5722), ((1, 'CB'), 0.4514), ((1, 'OG1'), -0.6764), ((1, 'CG2'), -0.2554), ((1, 'H'), 0.1934), ((1, 'HA'), 0.1087), ((1, 'HB'), -0.0323)]


In [12]:
max_atoms = 200
# positives
output_dir = "cholesterol-rdkit-fpocket-5A/positive"
os.makedirs(output_dir, exist_ok=True)

positive_files = glob.glob("filtered-rdkit-fpocket-5A/positive/*.pdb")
positive_files = sorted(positive_files, key=natural_sort_key)

for file in positive_files:
    pdb_df = pdb_to_dataframe(file)
    coords, encoded_matrix = rdkit_coords_and_encoding(file, pdb_df, pqr_cache=pqr_cache)
    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Failed encoding for {file}")
        continue
    inverse_distance = compute_inverse_pairwise_distances_from_coords(coords)

    if inverse_distance.shape[0] != encoded_matrix.shape[0]:
        raise ValueError(
            f"Atom mismatch in {file}: "
            f"inverse_distance={inverse_distance.shape}, "
            f"encoded_matrix={encoded_matrix.shape}"
        )

    combined_matrix = inverse_distance @ encoded_matrix

    num_atoms = inverse_distance.shape[0]

    if num_atoms > max_atoms:
        print(f"{file} has {num_atoms} atoms, exceeding the limit of {max_atoms}")
        raise Exception("Too many atoms!")

    combined_matrix = np.pad(
        combined_matrix,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode='constant'
    )

    base_name = os.path.splitext(os.path.basename(file))[0]
    output_path = os.path.join(output_dir, f"{base_name}_combined_matrix.npy")
    np.save(output_path, combined_matrix)

    #print(f"Saved: {output_path}")

[WARNING] Atom count mismatch for filtered-rdkit-fpocket-5A/positive/5NM4-filtered.pdb: pdb_df has 72 rows, RDKit non-H atoms has 67
[SKIP] Failed encoding for filtered-rdkit-fpocket-5A/positive/5NM4-filtered.pdb


In [13]:
max_atoms = 200
# unlabeleds
output_dir = "cholesterol-rdkit-fpocket-5A/unlabeled"
os.makedirs(output_dir, exist_ok=True)

unlabeled_files = glob.glob("filtered-rdkit-fpocket-5A/unlabeled/*.pdb")
unlabeled_files = sorted(unlabeled_files, key=natural_sort_key)

for file in unlabeled_files:
    pdb_df = pdb_to_dataframe(file)
    coords, encoded_matrix = rdkit_coords_and_encoding(file, pdb_df, pqr_cache=pqr_cache)
    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Failed encoding for {file}")
        continue
    inverse_distance = compute_inverse_pairwise_distances_from_coords(coords)

    if inverse_distance.shape[0] != encoded_matrix.shape[0]:
        raise ValueError(
            f"Atom mismatch in {file}: "
            f"inverse_distance={inverse_distance.shape}, "
            f"encoded_matrix={encoded_matrix.shape}"
        )

    combined_matrix = inverse_distance @ encoded_matrix

    num_atoms = inverse_distance.shape[0]

    if num_atoms > max_atoms:
        print(f"{file} has {num_atoms} atoms, exceeding the limit of {max_atoms}")
        #raise Exception("Too many atoms!")
        continue

    combined_matrix = np.pad(
        combined_matrix,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode='constant'
    )

    base_name = os.path.splitext(os.path.basename(file))[0]
    output_path = os.path.join(output_dir, f"{base_name}_combined_matrix.npy")
    np.save(output_path, combined_matrix)

    #print(f"Saved: {output_path}")

filtered-rdkit-fpocket-5A/unlabeled/2RH1-p35.pdb has 290 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/3A3Y-p96.pdb has 254 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/3AM6-p69.pdb has 283 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/3D4S-p35.pdb has 201 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/3NY9-p38.pdb has 230 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/3NYA-p11.pdb has 274 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/3NYA-p20.pdb has 222 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/3WGU-p85.pdb has 216 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/3WGV-p88.pdb has 241 atoms, exceeding the limit of 200
[WARNING] Atom count mismatch for filtered-rdkit-fpocket-5A/unlabeled/4BOE-p1.pdb: pdb_df has 316 rows, RDKit non-H atoms has 313
[SKIP] Failed encoding for filtered-rdkit-fpocket-5

[11:11:57] Explicit valence for atom # 51 C, 5, is greater than permitted


[WARN] RDKit sanitize warning for: filtered-rdkit-fpocket-5A/unlabeled/4XPF-p50.pdb


[11:11:58] Explicit valence for atom # 40 O, 3, is greater than permitted


[WARN] RDKit sanitize warning for: filtered-rdkit-fpocket-5A/unlabeled/4XPG-p38.pdb
filtered-rdkit-fpocket-5A/unlabeled/4XPH-p3.pdb has 207 atoms, exceeding the limit of 200
[WARNING] Atom count mismatch for filtered-rdkit-fpocket-5A/unlabeled/4XPH-p6.pdb: pdb_df has 46 rows, RDKit non-H atoms has 41
[SKIP] Failed encoding for filtered-rdkit-fpocket-5A/unlabeled/4XPH-p6.pdb
filtered-rdkit-fpocket-5A/unlabeled/4XPH-p47.pdb has 218 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/4XPT-p38.pdb has 577 atoms, exceeding the limit of 200
[WARNING] Atom count mismatch for filtered-rdkit-fpocket-5A/unlabeled/5C1M-p9.pdb: pdb_df has 59 rows, RDKit non-H atoms has 51
[SKIP] Failed encoding for filtered-rdkit-fpocket-5A/unlabeled/5C1M-p9.pdb
[WARNING] Atom count mismatch for filtered-rdkit-fpocket-5A/unlabeled/5C1M-p15.pdb: pdb_df has 54 rows, RDKit non-H atoms has 43
[SKIP] Failed encoding for filtered-rdkit-fpocket-5A/unlabeled/5C1M-p15.pdb
filtered-rdkit-fpocket-5A/unlabel

...

[WARN] RDKit sanitize warning for: filtered-rdkit-fpocket-5A/unlabeled/7WQ4-p11.pdb
filtered-rdkit-fpocket-5A/unlabeled/7WQ4-p61.pdb has 202 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/7WU4-p1.pdb has 304 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/7WU4-p56.pdb has 221 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/7WU5-p52.pdb has 222 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/7WYT-p127.pdb has 257 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/7WYT-p172.pdb has 217 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/7WYU-p114.pdb has 250 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/7WYU-p118.pdb has 250 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/7WYU-p219.pdb has 225 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/7WYU-p220.pdb has 225 atoms, exceeding the limit of 200
f

[11:18:06] Explicit valence for atom # 57 O, 3, is greater than permitted
[11:18:06] Explicit valence for atom # 11 O, 3, is greater than permitted


[WARN] RDKit sanitize warning for: filtered-rdkit-fpocket-5A/unlabeled/8XC1-p40.pdb
filtered-rdkit-fpocket-5A/unlabeled/8XC1-p44.pdb has 214 atoms, exceeding the limit of 200
[WARN] RDKit sanitize warning for: filtered-rdkit-fpocket-5A/unlabeled/8XC1-p51.pdb


[11:18:06] Explicit valence for atom # 33 C, 5, is greater than permitted
[11:18:06] Explicit valence for atom # 2 C, 5, is greater than permitted


[WARN] RDKit sanitize warning for: filtered-rdkit-fpocket-5A/unlabeled/8XC1-p74.pdb
[WARN] RDKit sanitize warning for: filtered-rdkit-fpocket-5A/unlabeled/8XC1-p79.pdb
filtered-rdkit-fpocket-5A/unlabeled/8XQJ-p1.pdb has 223 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/8XQL-p1-positive.pdb has 201 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/8XQL-p2.pdb has 318 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/8XQL-p50.pdb has 206 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/8XQL-p59.pdb has 232 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/8XQL-p70.pdb has 225 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/8XQP-p1.pdb has 507 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/8XQP-p24.pdb has 231 atoms, exceeding the limit of 200
filtered-rdkit-fpocket-5A/unlabeled/8XQS-p9.pdb has 221 atoms, exceeding the limit of 200
filtered-

In [14]:
feature_names = [
    'C', 'CA', 'CB', 'CD', 'CD1', 'CD2', 'CE', 'CE1', 'CE2', 'CE3',
    'CG', 'CG1', 'CG2', 'CH2', 'CZ', 'CZ2', 'CZ3',

    # Oxygen (8)
    'O', 'OH', 'OD1', 'OD2', 'OE1', 'OE2', 'OG', 'OG1',

    # Nitrogen (9)
    'N', 'NE', 'NE1', 'NE2', 'ND1', 'ND2', 'NZ', 'NH1', 'NH2',

    # Sulfur (2)
    'SD', 'SG',

    # Unknown (1)
    'UNKNOWN',

    "hybridization_SP",                         
    "hybridization_SP2",                        
    "hybridization_SP3",                        

    "partial_charge",                    

    "is_in_ring",                               
    "is_aromatic",                              

    "residue_acidic_ASP_GLU",                   
    "residue_basic_LYS_ARG",                    
    "residue_HIS",                              
    "residue_CYS",                              
    "residue_polar_ASN_GLN_SER_THR",            
    "residue_GLY",                              
    "residue_PRO",                              
    "residue_aromatic_PHE_TYR_TRP",             
    "residue_hydrophobic_ALA_ILE_LEU_MET_VAL"   
]

In [15]:
max_atoms = 200
# positives
output_dir = "cholesterol-rdkit-fpocket-5A/positive"
np.set_printoptions(threshold=np.inf)

positive_files = glob.glob("filtered-rdkit-fpocket-5A/positive/*.pdb")
positive_files = sorted(positive_files, key=natural_sort_key)

# Trackers
feature_usage = None                 # total atom-level usage
feature_file_presence = None         # per-file presence
num_features = None
total_atoms_all_files = 0
total_files = len(positive_files)

for file in positive_files:
    pdb_df = pdb_to_dataframe(file)
    coords, encoded_matrix = rdkit_coords_and_encoding(file, pdb_df, pqr_cache=pqr_cache)
    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Failed encoding for {file}")
        continue
    inverse_distance = compute_inverse_pairwise_distances_from_coords(coords)

    if inverse_distance.shape[0] != encoded_matrix.shape[0]:
        raise ValueError(
            f"Atom mismatch in {file}: "
            f"inverse_distance={inverse_distance.shape}, "
            f"encoded_matrix={encoded_matrix.shape}"
        )

    # Initialize trackers
    if feature_usage is None:
        num_features = encoded_matrix.shape[1]
        feature_usage = np.zeros(num_features, dtype=np.int64)
        feature_file_presence = np.zeros(num_features, dtype=np.int64)

    num_atoms = inverse_distance.shape[0]
    total_atoms_all_files += num_atoms

    # ---- Atom-level usage ----
    feature_usage += (encoded_matrix != 0).sum(axis=0).astype(np.int64)

    # ---- File-level presence ----
    feature_file_presence += (encoded_matrix != 0).any(axis=0).astype(np.int64)

    combined_matrix = inverse_distance @ encoded_matrix

    if num_atoms > max_atoms:
        print(f"{file} has {num_atoms} atoms, exceeding the limit of {max_atoms}")
        raise Exception("Too many atoms!")

    combined_matrix = np.pad(
        combined_matrix,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode='constant'
    )

# ---- AFTER LOOP ----

print("\n=== FEATURE USAGE SUMMARY ===")
print(f"Total atoms: {total_atoms_all_files}")
print(f"Total files: {total_files}\n")

for i in range(num_features):
    atom_pct = (feature_usage[i] / total_atoms_all_files) * 100 if total_atoms_all_files > 0 else 0
    file_pct = (feature_file_presence[i] / total_files) * 100 if total_files > 0 else 0

    print(
        f"{i:2d} | {feature_names[i]:40s} | "
        f"atoms: {feature_usage[i]:8d} ({atom_pct:7.3f}%) | "
        f"files: {feature_file_presence[i]:4d}/{total_files} ({file_pct:6.2f}%)"
    )

# ---- UNUSED FEATURES ----

unused_features = np.where(feature_usage == 0)[0]

print("\n=== UNUSED FEATURES ===")
for i in unused_features:
    print(f"{i:2d} | {feature_names[i]}")

[WARNING] Atom count mismatch for filtered-rdkit-fpocket-5A/positive/5NM4-filtered.pdb: pdb_df has 72 rows, RDKit non-H atoms has 67
[SKIP] Failed encoding for filtered-rdkit-fpocket-5A/positive/5NM4-filtered.pdb

=== FEATURE USAGE SUMMARY ===
Total atoms: 32804
Total files: 770

 0 | C                                        | atoms:     3949 ( 12.038%) | files:  769/770 ( 99.87%)
 1 | CA                                       | atoms:     3949 ( 12.038%) | files:  769/770 ( 99.87%)
 2 | CB                                       | atoms:     3695 ( 11.264%) | files:  769/770 ( 99.87%)
 3 | CD                                       | atoms:      352 (  1.073%) | files:  302/770 ( 39.22%)
 4 | CD1                                      | atoms:     2029 (  6.185%) | files:  729/770 ( 94.68%)
 5 | CD2                                      | atoms:     1528 (  4.658%) | files:  677/770 ( 87.92%)
 6 | CE                                       | atoms:      155 (  0.473%) | files:  131/770 ( 17.01%

In [ ]:
max_atoms = 200
# unlabeleds
output_dir = "cholesterol-rdkit-fpocket-5A/unlabeled"
np.set_printoptions(threshold=np.inf)

unlabeled_files = glob.glob("filtered-rdkit-fpocket-5A/unlabeled/*.pdb")
unlabeled_files = sorted(unlabeled_files, key=natural_sort_key)

# Trackers
feature_usage = None                 # total atom-level usage
feature_file_presence = None         # per-file presence
num_features = None
total_atoms_all_files = 0
total_files = len(unlabeled_files)

for file in unlabeled_files:
    pdb_df = pdb_to_dataframe(file)
    coords, encoded_matrix = rdkit_coords_and_encoding(file, pdb_df, pqr_cache=pqr_cache)
    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Failed encoding for {file}")
        continue
    inverse_distance = compute_inverse_pairwise_distances_from_coords(coords)

    if inverse_distance.shape[0] != encoded_matrix.shape[0]:
        raise ValueError(
            f"Atom mismatch in {file}: "
            f"inverse_distance={inverse_distance.shape}, "
            f"encoded_matrix={encoded_matrix.shape}"
        )

    # Initialize trackers
    if feature_usage is None:
        num_features = encoded_matrix.shape[1]
        feature_usage = np.zeros(num_features, dtype=np.int64)
        feature_file_presence = np.zeros(num_features, dtype=np.int64)

    num_atoms = inverse_distance.shape[0]
    total_atoms_all_files += num_atoms

    # ---- Atom-level usage ----
    feature_usage += (encoded_matrix != 0).sum(axis=0).astype(np.int64)

    # ---- File-level presence ----
    feature_file_presence += (encoded_matrix != 0).any(axis=0).astype(np.int64)

    combined_matrix = inverse_distance @ encoded_matrix

    if num_atoms > max_atoms:
        print(f"{file} has {num_atoms} atoms, exceeding the limit of {max_atoms}")
        #raise Exception("Too many atoms!")
        continue

    combined_matrix = np.pad(
        combined_matrix,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode='constant'
    )

# ---- AFTER LOOP ----

print("\n=== FEATURE USAGE SUMMARY ===")
print(f"Total atoms: {total_atoms_all_files}")
print(f"Total files: {total_files}\n")

for i in range(num_features):
    atom_pct = (feature_usage[i] / total_atoms_all_files) * 100 if total_atoms_all_files > 0 else 0
    file_pct = (feature_file_presence[i] / total_files) * 100 if total_files > 0 else 0

    print(
        f"{i:2d} | {feature_names[i]:40s} | "
        f"atoms: {feature_usage[i]:8d} ({atom_pct:7.3f}%) | "
        f"files: {feature_file_presence[i]:4d}/{total_files} ({file_pct:6.2f}%)"
    )

# ---- UNUSED FEATURES ----

unused_features = np.where(feature_usage == 0)[0]

print("\n=== UNUSED FEATURES ===")
for i in unused_features:
    print(f"{i:2d} | {feature_names[i]}")

In [20]:
import os
import glob
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 200)

def build_atom_level_feature_table(file_path, pqr_cache=None):
    """
    Returns a dataframe with:
      - original atom information from the PDB
      - active RDKit/RDKit-derived encoded features for each atom
    """
    pdb_df = pdb_to_dataframe(file_path).copy()
    coords, encoded_matrix = rdkit_coords_and_encoding(file_path, pdb_df, pqr_cache=pqr_cache)

    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Could not encode: {file_path}")
        return None, None, None

    if len(pdb_df) != encoded_matrix.shape[0]:
        print(f"[SKIP] Atom count mismatch for {file_path}: pdb_df={len(pdb_df)}, encoded={encoded_matrix.shape[0]}")
        return None, None, None

    atom_table = pdb_df.copy()

    # Add coordinate columns actually used by RDKit encoding
    atom_table["RDKit_X"] = coords[:, 0]
    atom_table["RDKit_Y"] = coords[:, 1]
    atom_table["RDKit_Z"] = coords[:, 2]

    # Add full feature vector columns
    for i, feat_name in enumerate(feature_names):
        atom_table[feat_name] = encoded_matrix[:, i]

    # Add easy-to-read active feature names per atom
    def active_features(row_vec):
        names = []
        for j, val in enumerate(row_vec):
            if feature_names[j] == "partial_charge":
                names.append(f"partial_charge={val:.4f}")
            elif val != 0:
                names.append(feature_names[j])
        return names

    atom_table["active_feature_names"] = [active_features(row) for row in encoded_matrix]

    return atom_table, coords, encoded_matrix


def summarize_feature_usage_over_files(file_list, pqr_cache=None):
    """
    Evaluates feature usage across a list of PDB files.
    Returns:
      - summary_df
      - per_file_results
    """
    feature_usage = np.zeros(len(feature_names), dtype=np.int64)
    feature_file_presence = np.zeros(len(feature_names), dtype=np.int64)

    total_atoms = 0
    processed_files = 0
    skipped_files = []

    per_file_results = []

    for file_path in file_list:
        pdb_df = pdb_to_dataframe(file_path)
        coords, encoded_matrix = rdkit_coords_and_encoding(file_path, pdb_df, pqr_cache=pqr_cache)

        if coords is None or encoded_matrix is None:
            skipped_files.append(file_path)
            continue

        if encoded_matrix.shape[1] != len(feature_names):
            skipped_files.append(file_path)
            continue

        processed_files += 1
        total_atoms += encoded_matrix.shape[0]

        feature_usage += (encoded_matrix != 0).sum(axis=0).astype(np.int64)
        feature_file_presence += (encoded_matrix != 0).any(axis=0).astype(np.int64)

        per_file_results.append({
            "file": file_path,
            "num_atoms": encoded_matrix.shape[0],
            "num_features_used": int((encoded_matrix != 0).any(axis=0).sum())
        })

    summary_rows = []
    for i, feat_name in enumerate(feature_names):
        atom_pct = (feature_usage[i] / total_atoms * 100) if total_atoms > 0 else 0.0
        file_pct = (feature_file_presence[i] / processed_files * 100) if processed_files > 0 else 0.0
        summary_rows.append({
            "feature_index": i,
            "feature_name": feat_name,
            "atom_count_nonzero": int(feature_usage[i]),
            "atom_percent_nonzero": atom_pct,
            "files_present": int(feature_file_presence[i]),
            "file_percent_present": file_pct
        })

    summary_df = pd.DataFrame(summary_rows).sort_values(
        by=["atom_count_nonzero", "files_present"],
        ascending=False
    ).reset_index(drop=True)

    print("=== RDKit FEATURE EVALUATION SUMMARY ===")
    print(f"Processed files: {processed_files}")
    print(f"Skipped files:   {len(skipped_files)}")
    print(f"Total atoms:     {total_atoms}")

    if skipped_files:
        print("\nSkipped examples:")
        for f in skipped_files[:10]:
            print(" -", f)

    return summary_df, pd.DataFrame(per_file_results), skipped_files


# -----------------------------
# Choose which files to evaluate
# -----------------------------
positive_files = sorted(glob.glob("filtered-rdkit-fpocket-5A/positive/*.pdb"), key=natural_sort_key)
unlabeled_files = sorted(glob.glob("filtered-rdkit-fpocket-5A/unlabeled/*.pdb"), key=natural_sort_key)

print(f"Positive files found:  {len(positive_files)}")
print(f"Unlabeled files found: {len(unlabeled_files)}")

# Evaluate all files
all_files = positive_files[:1]
feature_summary_df, per_file_df, skipped_files = summarize_feature_usage_over_files(all_files, pqr_cache=pqr_cache)

print("\n=== Top feature usage summary ===")
display(feature_summary_df.head(25))

print("\n=== Per-file summary ===")
display(per_file_df.head(20))

unused_df = feature_summary_df[feature_summary_df["atom_count_nonzero"] == 0].copy()
print("\n=== Unused features ===")
display(unused_df if not unused_df.empty else pd.DataFrame({"message": ["No completely unused features found."]}))

# -----------------------------
# Show one example file in detail
# -----------------------------
example_file = positive_files[0] if len(positive_files) > 0 else (unlabeled_files[0] if len(unlabeled_files) > 0 else None)

if example_file is not None:
    print(f"\n=== Detailed atom-level view for: {example_file} ===")
    atom_feature_table, coords, encoded_matrix = build_atom_level_feature_table(example_file, pqr_cache=pqr_cache)

    if atom_feature_table is not None:
        # Show original PDB atoms + important encoded columns
        preview_cols = [
            "Atom Name", "Residue Name", "Residue ID", "Chain ID",
            "X", "Y", "Z",
            "active_feature_names"
        ]

        preview_cols = [c for c in preview_cols if c in atom_feature_table.columns]
        display(atom_feature_table[preview_cols].head(50))

        print("\n=== Full atom-level table with all encoded features available in `atom_feature_table` ===")
        print(f"Shape: {atom_feature_table.shape}")

        # Optional save
        out_csv = os.path.splitext(os.path.basename(example_file))[0] + "_atom_level_rdkit_features.csv"
        atom_feature_table.to_csv(out_csv, index=False)
        print(f"Saved atom-level feature table to: {out_csv}")
else:
    print("No filtered PDB files were found.")

Positive files found:  770
Unlabeled files found: 72770
=== RDKit FEATURE EVALUATION SUMMARY ===
Processed files: 1
Skipped files:   0
Total atoms:     40

=== Top feature usage summary ===


,feature_index,feature_name,atom_count_nonzero,atom_percent_nonzero,files_present,file_percent_present
0,40,partial_charge,40,100.0,1,100.0
1,51,residue_hydrophobic_ALA_ILE_LEU_MET_VAL,28,70.0,1,100.0
2,39,hybridization_SP3,21,52.5,1,100.0
3,38,hybridization_SP2,19,47.5,1,100.0
4,50,residue_aromatic_PHE_TYR_TRP,12,30.0,1,100.0
5,41,is_in_ring,6,15.0,1,100.0
6,42,is_aromatic,6,15.0,1,100.0
7,0,C,5,12.5,1,100.0
8,1,CA,5,12.5,1,100.0
9,2,CB,5,12.5,1,100.0



=== Per-file summary ===


,file,num_atoms,num_features_used
0,filtered-rdkit-fpocket-5A/positive/1LRI-filtered.pdb,40,23



=== Unused features ===


,feature_index,feature_name,atom_count_nonzero,atom_percent_nonzero,files_present,file_percent_present
23,3,CD,0,0.0,0,0.0
24,9,CE3,0,0.0,0,0.0
25,13,CH2,0,0.0,0,0.0
26,15,CZ2,0,0.0,0,0.0
27,16,CZ3,0,0.0,0,0.0
28,19,OD1,0,0.0,0,0.0
29,20,OD2,0,0.0,0,0.0
30,21,OE1,0,0.0,0,0.0
31,22,OE2,0,0.0,0,0.0
32,23,OG,0,0.0,0,0.0



=== Detailed atom-level view for: filtered-rdkit-fpocket-5A/positive/1LRI-filtered.pdb ===


,Atom Name,Residue Name,Residue ID,Chain ID,X,Y,Z,active_feature_names
0,N,LEU,15,A,9.387,9.230000,15.660000,"[N, hybridization_SP3, partial_charge=-0.4157, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"
1,CA,LEU,15,A,8.792,10.132000,14.690000,"[CA, hybridization_SP3, partial_charge=-0.0518, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"
2,C,LEU,15,A,7.748,11.037000,15.308000,"[C, hybridization_SP2, partial_charge=0.5973, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"
3,O,LEU,15,A,7.390,12.069000,14.749000,"[O, hybridization_SP2, partial_charge=-0.5679, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"
4,CB,LEU,15,A,8.199,9.319000,13.531000,"[CB, hybridization_SP3, partial_charge=-0.1102, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"
5,CG,LEU,15,A,9.192,8.698000,12.566000,"[CG, hybridization_SP3, partial_charge=0.3531, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"
6,CD1,LEU,15,A,8.533,7.640000,11.681000,"[CD1, hybridization_SP3, partial_charge=-0.4121, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"
7,CD2,LEU,15,A,9.819,9.763000,11.664000,"[CD2, hybridization_SP3, partial_charge=-0.4121, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"
8,N,VAL,16,A,7.260,10.744000,16.514000,"[N, hybridization_SP2, partial_charge=-0.4157, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"
9,CA,VAL,16,A,6.234,11.603000,17.127001,"[CA, hybridization_SP3, partial_charge=-0.0875, residue_hydrophobic_ALA_ILE_LEU_MET_VAL]"



=== Full atom-level table with all encoded features available in `atom_feature_table` ===
Shape: (40, 63)
Saved atom-level feature table to: 1LRI-filtered_atom_level_rdkit_features.csv
